Quiz 49 — Grocery Store Optimisation  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np

np.random.seed(120)
sales       = np.random.randint(100, 1000, (6, 4, 8)).astype(float)
cost_price  = np.array([1.5, 2.0, 3.5, 0.8, 5.0, 1.2])
sell_price  = np.array([2.5, 3.2, 5.0, 1.5, 7.5, 2.0])

# ── 1. Profit grid ────────────────────────────────────────────────────────────
profit_pu = sell_price - cost_price                  # (6,)
# profit_pu (6,) → reshape to (6,1,1) to broadcast over (6,4,8)
profit_grid = sales * profit_pu[:, np.newaxis, np.newaxis]
total_profit_base = profit_grid.sum()
print(f"Total base profit: £{total_profit_base:,.2f}")

# ── 2. Profit per store ───────────────────────────────────────────────────────
store_profit = profit_grid.sum(axis=(0, 2))          # (4,) — sum cats + weeks
store_rank   = np.argsort(store_profit)[::-1]
print("\nStore ranking (profit):")
for r, s in enumerate(store_rank):
    print(f"  Rank {r+1}: Store {s+1}  £{store_profit[s]:,.2f}")

# ── 3. Profit per category ────────────────────────────────────────────────────
cat_profit = profit_grid.sum(axis=(1, 2))            # (6,)
best_cat   = np.argmax(cat_profit)
print(f"\nMost profitable category: Cat {best_cat+1} (£{cat_profit[best_cat]:,.2f})")

# ── 4. Underperforming store-week combos ─────────────────────────────────────
sw_sales = sales.sum(axis=0)                         # (4,8) — total per store-week
under    = np.argwhere(sw_sales < 1500)              # list of [store, week] pairs
print(f"\nUnderperforming store-week combos: {len(under)}")

# ── 5. Promotional uplift on top-2 categories, weeks 5-8 ─────────────────────
top2_cats = np.argsort(cat_profit)[-2:]
promoted  = sales.copy()
promoted[top2_cats, :, 4:] *= 1.15                  # weeks 5-8 = index 4:8

promo_profit_grid  = promoted * profit_pu[:, np.newaxis, np.newaxis]
total_profit_promo = promo_profit_grid.sum()

improvement = (total_profit_promo - total_profit_base) / total_profit_base * 100
print(f"\nProfit before promo: £{total_profit_base:,.2f}")
print(f"Profit after  promo: £{total_profit_promo:,.2f}")
print(f"Improvement        : {improvement:.2f}%")

# ── WHY ───────────────────────────────────────────────────────────────────────
# profit_pu[:, np.newaxis, np.newaxis] reshapes (6,) → (6,1,1).
# This broadcasts across stores (axis 1) and weeks (axis 2) automatically.
# sales[top2_cats, :, 4:] uses fancy indexing on axis 0, : on axis 1,
# and a slice on axis 2 — all three axes can be indexed simultaneously.